In [7]:
"""
Protocol-grade benchmark: AGS v3.7 vs Optuna TPE, SMAC3, Hyperopt,
RandomizedSearchCV, and HalvingGridSearchCV.

PRIOR WORK FRAMING (for the paper):
  • SMAC (Lindauer et al., JMLR 23, 2022) is the closest relative.
  • TuRBO (Eriksson et al., NeurIPS 2019) echoes AGS's trust-region logic.
  • F-Race (Birattari et al., 2010) is the racing lineage AGS descends from.
  AGS's contribution is the *combination* for discrete/conditional grids
  with fold-level racing, not the individual pieces.

PROTOCOL:
  • 20 seeds per problem.
  • Eval-equalized budgets (30, 60) AND time-equalized (TIME_BUDGET).
  • 95% confidence intervals.
  • Regret curves at evaluation checkpoints.
  • A TRUE conditional-space task (SVM kernel -> degree/gamma).

Run:
    pip install smac hyperopt scikit-learn optuna scipy
    python protocol_benchmark.py
"""
import time, warnings, math
from statistics import mean
warnings.filterwarnings("ignore")

import numpy as np

# CRITICAL: enable_halving_search_cv must be imported BEFORE HalvingGridSearchCV
from sklearn.experimental import enable_halving_search_cv  # noqa: F401

from sklearn.base import clone
from sklearn.datasets import (load_breast_cancer, load_digits, load_iris,
                              make_classification, make_moons)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (cross_val_score, RandomizedSearchCV,
                                     HalvingGridSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import optuna
from ags import AdaptiveGreedySearch

optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from smac import HyperparameterOptimizationFacade, Scenario
    from ConfigSpace import (ConfigurationSpace, CategoricalHyperparameter,
                             EqualsCondition, InCondition)
    HAS_SMAC = True
    _SMAC_ERR = ""
except Exception as _e:
    HAS_SMAC = False
    _SMAC_ERR = f"{type(_e).__name__}: {_e}"

try:
    from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
    HAS_HYPEROPT = True
    _HOPT_ERR = ""
except Exception as _e:
    HAS_HYPEROPT = False
    _HOPT_ERR = f"{type(_e).__name__}: {_e}"

try:
    from scipy.stats import t as t_dist, wilcoxon
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False


# ============================================================
# CONFIG
# ============================================================
QUICK_MODE   = False
CV           = 5
BUDGETS      = [30, 60]
TIME_BUDGET  = 60.0
N_SEEDS      = 20
N_JOBS       = -1
SEED_BASE    = 0

if QUICK_MODE:
    BUDGETS, N_SEEDS, TIME_BUDGET = [30], 1, 15.0


# ============================================================
# HELPERS
# ============================================================
def ci95(values):
    n = len(values)
    m = mean(values)
    if n < 2:
        return m, 0.0
    s = np.std(values, ddof=1)
    h = (t_dist.ppf(0.975, n - 1) * s / math.sqrt(n)
         if HAS_SCIPY else 1.96 * s / math.sqrt(n))
    return m, h


def ags_grab(obj, names, default=None):
    for n in names:
        if hasattr(obj, n):
            v = getattr(obj, n)
            if v is not None:
                return v
    return default


def _coerce_score(x):
    """Coerce any AGS internal record into a float score, or return None."""
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    if isinstance(x, dict):
        for k in ("score", "best_score", "value", "result",
                  "fitness", "objective", "loss"):
            if k in x:
                try:
                    return float(x[k])
                except (TypeError, ValueError):
                    pass
        for v in x.values():
            if isinstance(v, (int, float, np.integer, np.floating)):
                return float(v)
        return None
    if isinstance(x, (tuple, list)) and x:
        for v in reversed(x):
            try:
                return float(v)
            except (TypeError, ValueError):
                continue
    return None


def extract_best_so_far(ags, budget, cv):
    """Return (best_so_far_history, final_best_score)."""
    for attr in ("evaluated", "values", "history"):
        raw = getattr(ags, attr, None)
        if raw is None:
            continue
        if isinstance(raw, dict):
            items = list(raw.values())
        elif isinstance(raw, (list, tuple)):
            items = list(raw)
        else:
            continue
        if not items:
            continue
        scores = [s for s in (_coerce_score(x) for x in items) if s is not None]
        if not scores:
            continue
        best, hist = -np.inf, []
        for v in scores:
            best = max(best, v)
            hist.append(best)
        final = float(ags_grab(ags, ["best_score", "best_score_"], hist[-1]))
        return hist, final
    final = float(ags_grab(ags, ["best_score", "best_score_"], 0.0))
    return [final], final


# ============================================================
# DATASETS
# ============================================================
def load_dataset(name):
    if name == "breast_cancer":
        d = load_breast_cancer(); return d.data, d.target
    if name == "digits":
        d = load_digits(); return d.data, d.target
    if name == "iris":
        d = load_iris(); return d.data, d.target
    if name == "moons":
        return make_moons(n_samples=600, noise=0.25, random_state=0)
    if name == "synth_hd":
        return make_classification(
            n_samples=600, n_features=25, n_informative=12,
            n_classes=3, n_clusters_per_class=2, random_state=0)
    if name == "noisy":
        return make_classification(
            n_samples=600, n_features=20, n_informative=10,
            flip_y=0.25, random_state=0)
    raise ValueError(name)


# ============================================================
# SPACES
# ============================================================
def flat_space(name):
    if name == "RF":
        return (RandomForestClassifier(random_state=0, n_jobs=1),
                {"n_estimators":      [50, 100, 200, 300],
                 "max_depth":         [3, 5, 7, 10, None],
                 "min_samples_split": [2, 5, 10],
                 "min_samples_leaf":  [1, 2, 4],
                 "max_features":      ["sqrt", "log2"]})
    if name == "MLP":
        return (make_pipeline(StandardScaler(),
                              MLPClassifier(random_state=0, max_iter=500,
                                            early_stopping=True)),
                {"mlpclassifier__hidden_layer_sizes": [(32,), (64,), (32, 32), (64, 64)],
                 "mlpclassifier__alpha":              [0.0001, 0.001, 0.01],
                 "mlpclassifier__learning_rate_init": [0.001, 0.01]})
    if name == "SVM_flat":
        return (make_pipeline(StandardScaler(), SVC(random_state=0)),
                {"svc__C":      [0.01, 0.1, 1, 10, 100],
                 "svc__gamma":  ["scale", 0.001, 0.01, 0.1],
                 "svc__kernel": ["linear", "rbf", "poly"],
                 "svc__degree": [2, 3, 4]})
    if name == "KNN":
        return (KNeighborsClassifier(),
                {"n_neighbors": [1, 3, 5, 7, 9, 15],
                 "weights":     ["uniform", "distance"],
                 "p":           [1, 2]})
    raise ValueError(name)


# ============================================================
# CONDITIONAL SPACE
# ============================================================
def conditional_svm_model():
    return make_pipeline(StandardScaler(), SVC(random_state=0))


def conditional_svm_params_optuna(trial):
    kernel = trial.suggest_categorical("kernel", ["linear", "rbf", "poly"])
    C = trial.suggest_categorical("C", [0.01, 0.1, 1, 10, 100])
    params = {"svc__kernel": kernel, "svc__C": C}
    if kernel in ("rbf", "poly"):
        params["svc__gamma"] = trial.suggest_categorical(
            "gamma", ["scale", 0.001, 0.01, 0.1])
    if kernel == "poly":
        params["svc__degree"] = trial.suggest_categorical("degree", [2, 3, 4])
    return params


def conditional_svm_params_hyperopt():
    return hp.choice("kernel", [
        {"svc__kernel": "linear", "svc__C": hp.choice("C_lin", [0.01, 0.1, 1, 10, 100])},
        {"svc__kernel": "rbf",
         "svc__C": hp.choice("C_rbf", [0.01, 0.1, 1, 10, 100]),
         "svc__gamma": hp.choice("gamma_rbf", ["scale", 0.001, 0.01, 0.1])},
        {"svc__kernel": "poly",
         "svc__C": hp.choice("C_poly", [0.01, 0.1, 1, 10, 100]),
         "svc__gamma": hp.choice("gamma_poly", ["scale", 0.001, 0.01, 0.1]),
         "svc__degree": hp.choice("degree_poly", [2, 3, 4])},
    ])


# ============================================================
# RUNNERS
# ============================================================
def run_ags(model, grid, X, y, budget, seed, scoring):
    t0 = time.perf_counter()
    ags = AdaptiveGreedySearch(
        estimator=model, param_grid=grid, cv=CV, scoring=scoring,
        max_evaluations=budget, n_jobs=N_JOBS, random_state=seed,
    )
    ags.fit(X, y)
    wall = time.perf_counter() - t0
    hist, best = extract_best_so_far(ags, budget, CV)
    saved = int(ags_grab(ags, ["folds_saved", "saved_folds"], 0))
    poss  = int(ags_grab(ags, ["total_folds_possible", "folds_possible"], budget * CV))
    return hist, best, wall, saved, poss


def run_optuna(model, grid, X, y, budget, seed, scoring, conditional=False):
    t0 = time.perf_counter()
    hist = []

    def obj(trial):
        if conditional:
            p = conditional_svm_params_optuna(trial)
        else:
            p = {k: trial.suggest_categorical(k, list(v)) for k, v in grid.items()}
        m = clone(model); m.set_params(**p)
        s = float(cross_val_score(m, X, y, cv=CV, scoring=scoring,
                                  n_jobs=N_JOBS, error_score="raise").mean())
        hist.append(s)
        return s

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=seed))
    try:
        study.optimize(obj, n_trials=budget, show_progress_bar=False,
                       timeout=TIME_BUDGET if TIME_BUDGET else None)
    except Exception:
        pass
    wall = time.perf_counter() - t0
    best, h = -np.inf, []
    for v in hist:
        best = max(best, v); h.append(best)
    return h, float(study.best_value) if study.best_value is not None else -np.inf, wall, 0, budget * CV


def run_smac(model, grid, X, y, budget, seed, scoring, conditional=False):
    if not HAS_SMAC:
        return None, None, None, 0, 0
    t0 = time.perf_counter()
    hist = []

    if conditional:
        cs = ConfigurationSpace()
        kernel = CategoricalHyperparameter("kernel", ["linear", "rbf", "poly"])
        C = CategoricalHyperparameter("C", [0.01, 0.1, 1, 10, 100])
        gamma = CategoricalHyperparameter("gamma", ["scale", 0.001, 0.01, 0.1])
        degree = CategoricalHyperparameter("degree", [2, 3, 4])
        cs.add([kernel, C, gamma, degree])
        cs.add(EqualsCondition(degree, kernel, "poly"))
        cs.add(InCondition(gamma, kernel, ["rbf", "poly"]))

        def train(config, seed=0):
            p = {"svc__kernel": config["kernel"], "svc__C": config["C"]}
            if config["kernel"] in ("rbf", "poly"):
                p["svc__gamma"] = config["gamma"]
            if config["kernel"] == "poly":
                p["svc__degree"] = config["degree"]
            m = clone(model); m.set_params(**p)
            s = float(cross_val_score(m, X, y, cv=CV, scoring=scoring,
                                      n_jobs=N_JOBS, error_score="raise").mean())
            hist.append(s)
            return -s
    else:
        cs = ConfigurationSpace()
        for k, vals in grid.items():
            cs.add(CategoricalHyperparameter(k, [str(v) for v in vals]))

        def train(config, seed=0):
            p = {}
            for k, vals in grid.items():
                chosen = config[k]
                p[k] = next(v for v in vals if str(v) == chosen)
            m = clone(model); m.set_params(**p)
            s = float(cross_val_score(m, X, y, cv=CV, scoring=scoring,
                                      n_jobs=N_JOBS, error_score="raise").mean())
            hist.append(s)
            return -s

    scenario = Scenario(cs, deterministic=True, n_trials=budget, seed=seed,
                        walltime_limit=TIME_BUDGET)
    smac = HyperparameterOptimizationFacade(scenario, train)
    try:
        smac.optimize()
    except Exception:
        pass
    wall = time.perf_counter() - t0
    best, h = -np.inf, []
    for v in hist:
        best = max(best, v); h.append(best)
    return h, float(best), wall, 0, budget * CV


def run_hyperopt(model, grid, X, y, budget, seed, scoring, conditional=False):
    if not HAS_HYPEROPT:
        return None, None, None, 0, 0
    t0 = time.perf_counter()
    hist = []

    if conditional:
        space = conditional_svm_params_hyperopt()
    else:
        space = {k: hp.choice(k, list(v)) for k, v in grid.items()}

    def obj(p):
        params = p if conditional else {k: p[k] for k in grid}
        m = clone(model); m.set_params(**params)
        s = float(cross_val_score(m, X, y, cv=CV, scoring=scoring,
                                  n_jobs=N_JOBS, error_score="raise").mean())
        hist.append(s)
        return {"loss": -s, "status": STATUS_OK}

    trials = Trials()
    try:
        fmin(obj, space, algo=tpe.suggest, max_evals=budget, trials=trials,
             rstate=np.random.default_rng(seed),
             timeout=TIME_BUDGET if TIME_BUDGET else None)
    except Exception:
        pass
    wall = time.perf_counter() - t0
    best, h = -np.inf, []
    for v in hist:
        best = max(best, v); h.append(best)
    return h, float(best), wall, 0, budget * CV


def run_random(model, grid, X, y, budget, seed, scoring):
    t0 = time.perf_counter()
    rs = RandomizedSearchCV(
        clone(model), grid, n_iter=budget, cv=CV, scoring=scoring,
        random_state=seed, n_jobs=N_JOBS, refit=False,
    )
    rs.fit(X, y)
    wall = time.perf_counter() - t0
    scores = rs.cv_results_["mean_test_score"]
    best, h = -np.inf, []
    for v in scores:
        best = max(best, v); h.append(best)
    return h, float(rs.best_score_), wall, 0, budget * CV


def run_halving(model, grid, X, y, budget, seed, scoring):
    t0 = time.perf_counter()
    try:
        hs = HalvingGridSearchCV(
            clone(model), grid, cv=CV, scoring=scoring,
            factor=3, min_resources="exhaust",
            n_candidates=budget, random_state=seed,
            n_jobs=N_JOBS, refit=False,
        )
        hs.fit(X, y)
        wall = time.perf_counter() - t0
        scores = hs.cv_results_["mean_test_score"]
        best, h = -np.inf, []
        for v in scores:
            best = max(best, v); h.append(best)
        return h, float(hs.best_score_), wall, 0, budget * CV
    except Exception:
        wall = time.perf_counter() - t0
        return None, None, wall, 0, 0


# ============================================================
# PROBLEM LIST
# ============================================================
PROBLEMS = [
    ("rf_digits",         "digits",        "RF",         "f1_macro", False),
    ("rf_breast_cancer",  "breast_cancer", "RF",         "f1_macro", False),
    ("mlp_digits",        "digits",        "MLP",        "f1_macro", False),
    ("svm_conditional",   "breast_cancer", "SVM_flat",   "f1_macro", True),
    ("svm_flat_wine",     "breast_cancer", "SVM_flat",   "f1_macro", False),
    ("knn_iris",          "iris",          "KNN",        "f1_macro", False),
    ("rf_moons",          "moons",         "RF",         "f1_macro", False),
    ("rf_noisy",          "noisy",         "RF",         "f1_macro", False),
]

if QUICK_MODE:
    PROBLEMS = PROBLEMS[:3]


# ============================================================
# MAIN
# ============================================================
print("=" * 100)
print("PROTOCOL BENCHMARK — AGS v3.7 vs Optuna / SMAC3 / Hyperopt / Random / Halving")
print(f"Seeds: {N_SEEDS}  |  Budgets: {BUDGETS}  |  Time cap: {TIME_BUDGET}s  |  CV: {CV}")
print(f"SMAC3: {HAS_SMAC}  |  Hyperopt: {HAS_HYPEROPT}  |  scipy: {HAS_SCIPY}")
if not HAS_SMAC:
    print(f"  SMAC3 error: {_SMAC_ERR}")
if not HAS_HYPEROPT:
    print(f"  Hyperopt error: {_HOPT_ERR}")
print("=" * 100)

try:
    _m, _g = flat_space("RF")
    _X, _y = load_dataset("iris")
    _p = AdaptiveGreedySearch(estimator=_m, param_grid=_g, cv=CV,
                              scoring="f1_macro", max_evaluations=2,
                              n_jobs=1, random_state=0)
    _p.fit(_X, _y)
    _hist, _best = extract_best_so_far(_p, 2, CV)
    print(f"AGS probe OK: best_score={_best}  history_len={len(_hist)}")
except Exception as _e:
    print(f"AGS probe FAILED: {type(_e).__name__}: {_e}")
print("=" * 100)

METHODS = ["AGS", "Optuna", "SMAC3", "Hyperopt", "Random", "Halving"]
store = {}

for pname, dsname, space_name, scoring, conditional in PROBLEMS:
    X, y = load_dataset(dsname)
    model, grid = flat_space(space_name)
    grid_size = int(np.prod([len(v) for v in grid.values()]))

    print(f"\n{'='*100}")
    print(f"PROBLEM: {pname}  |  dataset={dsname} ({X.shape[0]}×{X.shape[1]})  "
          f"|  space={space_name} ({grid_size} configs)  |  conditional={conditional}")
    print(f"{'='*100}")

    for budget in BUDGETS:
        for method in METHODS:
            store[(pname, budget, method)] = {"scores": [], "times": [], "histories": []}

            for s in range(N_SEEDS):
                seed = SEED_BASE + s
                try:
                    if method == "AGS":
                        h, best, wall, _, _ = run_ags(model, grid, X, y, budget, seed, scoring)
                    elif method == "Optuna":
                        h, best, wall, _, _ = run_optuna(model, grid, X, y, budget, seed, scoring, conditional)
                    elif method == "SMAC3":
                        h, best, wall, _, _ = run_smac(model, grid, X, y, budget, seed, scoring, conditional)
                    elif method == "Hyperopt":
                        h, best, wall, _, _ = run_hyperopt(model, grid, X, y, budget, seed, scoring, conditional)
                    elif method == "Random":
                        h, best, wall, _, _ = run_random(model, grid, X, y, budget, seed, scoring)
                    elif method == "Halving":
                        h, best, wall, _, _ = run_halving(model, grid, X, y, budget, seed, scoring)
                    else:
                        continue
                    if h is None or best is None:
                        continue
                    store[(pname, budget, method)]["scores"].append(float(best))
                    store[(pname, budget, method)]["times"].append(float(wall))
                    store[(pname, budget, method)]["histories"].append(h)
                except Exception as e:
                    print(f"    {method} seed {seed} FAILED: {type(e).__name__}: {e}")

        print(f"\n  BUDGET {budget}:")
        print(f"    {'Method':<10}{'Mean':>10}{'±95%CI':>10}{'Wall(s)':>10}{'N':>5}")
        print(f"    {'-'*45}")
        for method in METHODS:
            rec = store[(pname, budget, method)]
            if not rec["scores"]:
                print(f"    {method:<10}{'—':>10}{'—':>10}{'—':>10}{0:>5}")
                continue
            m, h = ci95(rec["scores"])
            print(f"    {method:<10}{m:>10.4f}{h:>10.4f}{mean(rec['times']):>10.1f}"
                  f"{len(rec['scores']):>5}")

print("\n" + "=" * 100)
print("AGGREGATE ACROSS PROBLEMS  (mean of per-problem means ± 95% CI)")
print("=" * 100)

for budget in BUDGETS:
    print(f"\n  BUDGET {budget}:")
    print(f"    {'Method':<10}{'Mean':>10}{'±95%CI':>10}{'Med Δ vs AGS':>14}{'Wall(s)':>10}")
    print(f"    {'-'*55}")

    per_problem = {m: [] for m in METHODS}
    for pname, _, _, _, _ in PROBLEMS:
        for method in METHODS:
            rec = store.get((pname, budget, method), None)
            if rec and rec["scores"]:
                per_problem[method].append(mean(rec["scores"]))

    ags_means = per_problem["AGS"]
    for method in METHODS:
        vals = per_problem[method]
        if not vals:
            print(f"    {method:<10}{'—':>10}{'—':>10}{'—':>14}{'—':>10}")
            continue
        m, h = ci95(vals)
        if method != "AGS" and ags_means and len(vals) == len(ags_means):
            deltas = [a - b for a, b in zip(ags_means, vals)]
            med_s = f"{float(np.median(deltas)):+.4f}"
        else:
            med_s = "—"
        walls = []
        for pname, _, _, _, _ in PROBLEMS:
            rec = store.get((pname, budget, method), None)
            if rec and rec["times"]:
                walls.append(mean(rec["times"]))
        print(f"    {method:<10}{m:>10.4f}{h:>10.4f}{med_s:>14}"
              f"{mean(walls) if walls else 0:>10.1f}")

print("\n" + "=" * 100)
print("REGRET CURVES  (mean best-so-far across seeds; higher = better)")
print("=" * 100)

CHECKPOINTS = [1, 5, 10, 20, 30, 60]

for pname, _, _, _, _ in PROBLEMS:
    for budget in BUDGETS:
        cps = [c for c in CHECKPOINTS if c <= budget]
        if not cps:
            continue
        print(f"\n  {pname}  (budget {budget})")
        header = f"    {'Method':<10}" + "".join(f"{c:>8}" for c in cps)
        print(header)
        print(f"    {'-'*len(header)}")
        for method in METHODS:
            rec = store[(pname, budget, method)]
            if not rec["histories"]:
                continue
            rows = []
            for h in rec["histories"]:
                rows.append([h[min(c, len(h)) - 1] if h else -np.inf for c in cps])
            arr = np.array(rows)
            print(f"    {method:<10}" + "".join(f"{v:>8.4f}" for v in arr.mean(axis=0)))

print("\n" + "=" * 100)
print("FINAL VERDICT")
print("=" * 100)
for budget in BUDGETS:
    ags_vals, opt_vals = [], []
    for pname, _, _, _, _ in PROBLEMS:
        a = store.get((pname, budget, "AGS"), None)
        o = store.get((pname, budget, "Optuna"), None)
        if a and o and a["scores"] and o["scores"]:
            ags_vals.append(mean(a["scores"]))
            opt_vals.append(mean(o["scores"]))
    if not ags_vals:
        continue
    deltas = np.array([a - o for a, o in zip(ags_vals, opt_vals)])
    print(f"\n  Budget {budget}: AGS vs Optuna across {len(deltas)} problems")
    print(f"    Mean Δ       : {deltas.mean():+.4f}")
    print(f"    Median Δ     : {float(np.median(deltas)):+.4f}")
    print(f"    AGS better   : {int((deltas > 0.001).sum())}/{len(deltas)}")
    print(f"    Optuna better: {int((deltas < -0.001).sum())}/{len(deltas)}")
    if HAS_SCIPY and len(deltas) >= 5:
        stat, p = wilcoxon(deltas)
        print(f"    Wilcoxon p   : {p:.4f}  "
              f"[{'SIGNIFICANT' if p < 0.05 else 'not significant'}]")

print("\nDone.")

PROTOCOL BENCHMARK — AGS v3.7 vs Optuna / SMAC3 / Hyperopt / Random / Halving
Seeds: 20  |  Budgets: [30, 60]  |  Time cap: 60.0s  |  CV: 5
SMAC3: True  |  Hyperopt: True  |  scipy: True
AGS probe OK: best_score=0.9393715449335318  history_len=2

PROBLEM: rf_digits  |  dataset=digits (1797×64)  |  space=RF (360 configs)  |  conditional=False
[INFO][abstract_initial_design.py:91] Reducing the number of initial configurations from 50 to 7 (max_ratio == 0.25).
[INFO][abstract_initial_design.py:143] Using 7 initial design configurations and 0 additional configurations.
[INFO][abstract_intensifier.py:313] Using only one seed for deterministic scenario.
[INFO][abstract_intensifier.py:523] Added config 7caa10 as new incumbent because there are no incumbents yet.
[INFO][abstract_intensifier.py:630] Added config f5cf13 and rejected config 7caa10 as incumbent because it is not better than the incumbents on 1 instances: 
[INFO][abstract_intensifier.py:630] Added config 90a3c8 and rejected config 

In [1]:
"""
Quick benchmark: AGS v3.7.1 (profiling=True) vs Random / Optuna / Hyperopt.
4 moderate problems, 5 seeds, budgets 30 and 60.
"""
import time, warnings
from statistics import mean
warnings.filterwarnings("ignore")
import numpy as np

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer, load_digits, load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import optuna
from ags import AdaptiveGreedySearch

optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
    HAS_HYPEROPT = True
except ImportError:
    HAS_HYPEROPT = False

try:
    from scipy.stats import t as t_dist
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False


# ============================================================
# CONFIG
# ============================================================
CV       = 5
BUDGETS  = [30, 60]
N_SEEDS  = 5
N_JOBS   = -1
SCORING  = "f1_macro"


def ci95(vals):
    n = len(vals)
    m = mean(vals)
    if n < 2:
        return m, 0.0
    s = np.std(vals, ddof=1)
    h = (t_dist.ppf(0.975, n-1) * s / np.sqrt(n) if HAS_SCIPY
         else 1.96 * s / np.sqrt(n))
    return m, h


# ============================================================
# DATASETS & SPACES
# ============================================================
def load_dataset(name):
    if name == "breast_cancer": d = load_breast_cancer(); return d.data, d.target
    if name == "digits":        d = load_digits();        return d.data, d.target
    if name == "wine":          d = load_wine();          return d.data, d.target
    raise ValueError(name)


def rf_space():
    return (RandomForestClassifier(random_state=0, n_jobs=1),
            {"n_estimators":      [50, 100, 200],
             "max_depth":         [3, 5, 7, 10, None],
             "min_samples_split": [2, 5, 10],
             "min_samples_leaf":  [1, 2, 4],
             "max_features":      ["sqrt", "log2"],
             "criterion":         ["gini", "entropy"],
             "bootstrap":         [True, False]})


def svm_space():
    return (make_pipeline(StandardScaler(), SVC(random_state=0)),
            {"svc__C":            [0.01, 0.1, 1, 10, 100, 1000],
             "svc__gamma":        ["scale", 0.001, 0.01, 0.1, 1.0],
             "svc__kernel":       ["linear", "rbf", "poly"],
             "svc__degree":       [2, 3, 4],
             "svc__class_weight": [None, "balanced"]})


def xgb_space():
    if HAS_XGB:
        model = XGBClassifier(random_state=0, n_jobs=1, verbosity=0)
    else:
        from sklearn.ensemble import HistGradientBoostingClassifier
        model = HistGradientBoostingClassifier(random_state=0)
    return (model,
            {"max_depth":        [3, 5, 7],
             "learning_rate":    [0.01, 0.05, 0.1, 0.3],
             "n_estimators":     [50, 100, 200],
             "subsample":        [0.6, 0.8, 1.0],
             "colsample_bytree": [0.6, 0.8, 1.0],
             "reg_alpha":        [0, 0.1, 1.0]})


SPACES = {"rf": rf_space, "svm": svm_space, "xgb": xgb_space}

PROBLEMS = [
    ("rf_breast_cancer",  "breast_cancer", "rf"),
    ("rf_digits",         "digits",        "rf"),
    ("svm_breast_cancer", "breast_cancer", "svm"),
    ("xgb_wine",          "wine",          "xgb"),
]


# ============================================================
# RUNNERS
# ============================================================
def run_random(model, grid, X, y, budget, seed):
    t0 = time.perf_counter()
    rs = RandomizedSearchCV(clone(model), grid, n_iter=budget, cv=CV,
                            scoring=SCORING, random_state=seed,
                            n_jobs=N_JOBS, refit=False)
    rs.fit(X, y)
    return float(rs.best_score_), time.perf_counter() - t0


def run_optuna(model, grid, X, y, budget, seed):
    t0 = time.perf_counter()
    def obj(trial):
        p = {k: trial.suggest_categorical(k, list(v)) for k, v in grid.items()}
        m = clone(model); m.set_params(**p)
        return float(cross_val_score(m, X, y, cv=CV, scoring=SCORING,
                                     n_jobs=N_JOBS, error_score="raise").mean())
    study = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(obj, n_trials=budget, show_progress_bar=False)
    return float(study.best_value), time.perf_counter() - t0


def run_hyperopt(model, grid, X, y, budget, seed):
    if not HAS_HYPEROPT:
        return None, None
    t0 = time.perf_counter()
    space = {k: hp.choice(k, list(v)) for k, v in grid.items()}
    def obj(p):
        m = clone(model); m.set_params(**p)
        s = float(cross_val_score(m, X, y, cv=CV, scoring=SCORING,
                                  n_jobs=N_JOBS, error_score="raise").mean())
        return {"loss": -s, "status": STATUS_OK}
    trials = Trials()
    fmin(obj, space, algo=tpe.suggest, max_evals=budget, trials=trials,
         rstate=np.random.default_rng(seed))
    best = min(t["result"]["loss"] for t in trials.trials
               if t["result"]["status"] == STATUS_OK)
    return -float(best), time.perf_counter() - t0


def run_ags(model, grid, X, y, budget, seed):
    t0 = time.perf_counter()
    kwargs = dict(estimator=model, param_grid=grid, cv=CV, scoring=SCORING,
                  max_evaluations=budget, n_jobs=N_JOBS, random_state=seed)
    try:
        ags = AdaptiveGreedySearch(**kwargs, profiling=True)
        used_profiling = True
    except TypeError:
        ags = AdaptiveGreedySearch(**kwargs)
        used_profiling = False
    ags.fit(X, y)
    elapsed = time.perf_counter() - t0
    score = getattr(ags, "best_score", None)
    if score is None:
        score = getattr(ags, "best_score_", None)
    return float(score), elapsed, ags, used_profiling


# ============================================================
# MAIN
# ============================================================
print("=" * 88)
print("AGS v3.7.1 (profiling=True) vs Random / Optuna / Hyperopt")
print(f"Seeds: {N_SEEDS}  |  Budgets: {BUDGETS}  |  CV: {CV}  |  Metric: {SCORING}")
print(f"Hyperopt available: {HAS_HYPEROPT}  |  XGBoost available: {HAS_XGB}")
print("=" * 88)

# Version check
try:
    import importlib.metadata
    print(f"Installed AGS: {importlib.metadata.version('adaptive-greedy-search')}")
except Exception:
    pass
print("=" * 88)

METHODS = ["AGS", "Optuna", "Hyperopt", "Random"]
store = {}
first_profiling = None
profiling_supported = None

for pname, dsname, sname in PROBLEMS:
    X, y = load_dataset(dsname)
    model, grid = SPACES[sname]()
    grid_size = int(np.prod([len(v) for v in grid.values()]))

    print(f"\n{'='*88}")
    print(f"PROBLEM: {pname}  |  {dsname} ({X.shape[0]}×{X.shape[1]})  "
          f"|  {grid_size} configs")
    print(f"{'='*88}")

    for budget in BUDGETS:
        for method in METHODS:
            store[(pname, budget, method)] = {"scores": [], "times": []}

            for s in range(N_SEEDS):
                try:
                    if method == "AGS":
                        sc, tm, ags_obj, used = run_ags(model, grid, X, y, budget, s)
                        if profiling_supported is None:
                            profiling_supported = used
                        if used and first_profiling is None:
                            first_profiling = ags_obj
                    elif method == "Optuna":
                        sc, tm = run_optuna(model, grid, X, y, budget, s)
                    elif method == "Hyperopt":
                        sc, tm = run_hyperopt(model, grid, X, y, budget, s)
                    elif method == "Random":
                        sc, tm = run_random(model, grid, X, y, budget, s)
                    else:
                        continue
                    if sc is None or tm is None:
                        continue
                    store[(pname, budget, method)]["scores"].append(sc)
                    store[(pname, budget, method)]["times"].append(tm)
                except Exception as e:
                    print(f"    {method} seed {s}: {type(e).__name__}: {e}")

        print(f"\n  BUDGET {budget}:")
        print(f"    {'Method':<12}{'Mean':>10}{'±95%CI':>10}"
              f"{'Wall(s)':>10}{'N':>5}")
        print(f"    {'-'*47}")
        for method in METHODS:
            rec = store[(pname, budget, method)]
            if not rec["scores"]:
                print(f"    {method:<12}{'—':>10}{'—':>10}{'—':>10}{0:>5}")
                continue
            m, h = ci95(rec["scores"])
            print(f"    {method:<12}{m:>10.4f}{h:>10.4f}"
                  f"{mean(rec['times']):>10.1f}{len(rec['scores']):>5}")


# ============================================================
# AGGREGATE
# ============================================================
print("\n" + "=" * 88)
print("AGGREGATE ACROSS PROBLEMS  (mean of per-problem means ± 95% CI)")
print("=" * 88)

for budget in BUDGETS:
    print(f"\n  BUDGET {budget}:")
    print(f"    {'Method':<12}{'Mean':>10}{'±95%CI':>10}"
          f"{'Med Δ vs AGS':>15}{'Wall(s)':>10}")
    print(f"    {'-'*57}")

    per_problem = {m: [] for m in METHODS}
    for pname, _, _ in PROBLEMS:
        for method in METHODS:
            rec = store.get((pname, budget, method))
            if rec and rec["scores"]:
                per_problem[method].append(mean(rec["scores"]))

    ags_means = per_problem["AGS"]
    for method in METHODS:
        vals = per_problem[method]
        if not vals:
            print(f"    {method:<12}{'—':>10}{'—':>10}{'—':>15}{'—':>10}")
            continue
        m, h = ci95(vals)
        if method != "AGS" and ags_means and len(vals) == len(ags_means):
            deltas = [a - b for a, b in zip(ags_means, vals)]
            med_s = f"{float(np.median(deltas)):+.4f}"
        else:
            med_s = "—"
        walls = []
        for pname, _, _ in PROBLEMS:
            rec = store.get((pname, budget, method))
            if rec and rec["times"]:
                walls.append(mean(rec["times"]))
        print(f"    {method:<12}{m:>10.4f}{h:>10.4f}{med_s:>15}"
              f"{mean(walls) if walls else 0:>10.1f}")


# ============================================================
# AGS PROFILING INFO (if exposed)
# ============================================================
print("\n" + "=" * 88)
print("AGS PROFILING  (v3.7.1 profiling=True)")
print("=" * 88)
print(f"  profiling=True accepted by AGS: {profiling_supported}")

if first_profiling is not None:
    for attr in ("profiling_report", "profiling_info", "profiling_stats",
                 "profiling", "profile"):
        if hasattr(first_profiling, attr):
            v = getattr(first_profiling, attr)
            if v is None or callable(v):
                continue
            if isinstance(v, dict):
                keys = list(v.keys())
                print(f"  AGS.{attr} (dict, {len(v)} entries) — first keys:")
                for k in keys[:10]:
                    try:
                        print(f"    {k}: {v[k]}")
                    except Exception:
                        pass
                if len(keys) > 10:
                    print(f"    ... ({len(keys)-10} more)")
            else:
                print(f"  AGS.{attr}: {v}")
            break
    else:
        print("  No profiling attribute exposed on fitted instance.")
else:
    print("  No AGS instance captured.")


# ============================================================
# HEAD-TO-HEAD
# ============================================================
print("\n" + "=" * 88)
print("HEAD-TO-HEAD  (AGS vs each method, per budget)")
print("=" * 88)

for budget in BUDGETS:
    print(f"\n  BUDGET {budget}:")
    for other in ["Optuna", "Hyperopt", "Random"]:
        ags_v, oth_v = [], []
        for pname, _, _ in PROBLEMS:
            a = store.get((pname, budget, "AGS"))
            o = store.get((pname, budget, other))
            if a and o and a["scores"] and o["scores"]:
                ags_v.append(mean(a["scores"]))
                oth_v.append(mean(o["scores"]))
        if not ags_v:
            continue
        d = np.array([a - o for a, o in zip(ags_v, oth_v)])
        print(f"    AGS vs {other:<10}  "
              f"mean Δ={d.mean():+.4f}  median Δ={float(np.median(d)):+.4f}  "
              f"AGS wins {int((d>0.001).sum())}/{len(d)}")

print("\nDone.")

AGS v3.7.1 (profiling=True) vs Random / Optuna / Hyperopt
Seeds: 5  |  Budgets: [30, 60]  |  CV: 5  |  Metric: f1_macro
Hyperopt available: True  |  XGBoost available: True
Installed AGS: 3.7.1

PROBLEM: rf_breast_cancer  |  breast_cancer (569×30)  |  1080 configs
100%|██████████| 30/30 [00:07<00:00,  3.96trial/s, best loss: -0.9678852224451611]

  BUDGET 30:
    Method            Mean    ±95%CI   Wall(s)    N
    -----------------------------------------------
    AGS             0.9622    0.0043      11.8    5
    Optuna          0.9641    0.0017       8.0    5
    Hyperopt        0.9649    0.0027       6.8    5
    Random          0.9660    0.0023       3.3    5
100%|██████████| 60/60 [00:16<00:00,  3.69trial/s, best loss: -0.9678852224451611]

  BUDGET 60:
    Method            Mean    ±95%CI   Wall(s)    N
    -----------------------------------------------
    AGS             0.9623    0.0043      16.8    5
    Optuna          0.9661    0.0016      15.2    5
    Hyperopt        0